In [1]:
import os
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import shutil
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

from RAG_Ingestion import ingest_pdf_generator
from retrieval import HybridRetriever


f:\Antigravity_folder\rag_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hybrid_retriever = HybridRetriever()

query = "How many attention heads does the Transformer use, and what is the dimension of each head? "

docs = hybrid_retriever.retrieve(query)

Setting up Vector Retriever...
Setting up BM25 Retriever...
Setting up Hybrid Retriever...
Hybrid Setup complete!


In [3]:
doc = docs[0]
original_data = json.loads(doc.metadata["original_content"])

In [4]:
original_data

{'raw_text': '3.2.3 Applications of Attention in our Model\n\nThe Transformer uses multi-head attention in three different ways:\n\n• In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [38, 2, 9].\n\n• The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.\n\n• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flo

In [5]:
[doc.metadata.get("source_file") for doc in docs]

for doc in docs:
    original_data = json.loads(chunk.metadata["original_content"])


NameError: name 'chunk' is not defined

In [7]:

prompt_text= ""
source = []
for i, chunk in enumerate(docs):
    prompt_text += f"--- Document {i+1} ---\n"
    
    if "original_content" in chunk.metadata:
        original_data = json.loads(chunk.metadata["original_content"])# convert the dumped json string back to dictionary
        
        source.append(original_data.get('source_file', 'Unknown'))
        # Add raw text
        raw_text = original_data.get("raw_text", "")
        if raw_text:
            prompt_text += f"TEXT:\n{raw_text}\n\n"
        
        # Add tables as HTML
        tables_html = original_data.get("tables_html", [])
        if tables_html:
            prompt_text += "TABLES:\n"
            for j, table in enumerate(tables_html):
                prompt_text += f"Table {j+1}:\n{table}\n\n"
    
    prompt_text += "\n"

source = list(set(source))    

In [15]:
from pprint import pprint
pprint(prompt_text)

('--- Document 1 ---\n'
 'TEXT:\n'
 '3.2.3 Applications of Attention in our Model\n'
 '\n'
 'The Transformer uses multi-head attention in three different ways:\n'
 '\n'
 '• In "encoder-decoder attention" layers, the queries come from the previous '
 'decoder layer, and the memory keys and values come from the output of the '
 'encoder. This allows every position in the decoder to attend over all '
 'positions in the input sequence. This mimics the typical encoder-decoder '
 'attention mechanisms in sequence-to-sequence models such as [38, 2, 9].\n'
 '\n'
 '• The encoder contains self-attention layers. In a self-attention layer all '
 'of the keys, values and queries come from the same place, in this case, the '
 'output of the previous layer in the encoder. Each position in the encoder '
 'can attend to all positions in the previous layer of the encoder.\n'
 '\n'
 '• Similarly, self-attention layers in the decoder allow each position in the '
 'decoder to attend to all positions in the